In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import numpy as np
from pathlib import Path
from pprint import pprint
from datetime import timedelta


path = Path().resolve().parent.parent / "data"

In [ ]:
# local dependencies
import conform_data as cd
import medcode_utils as mu

# Environment Params

In [4]:
ICU_TRANSFER_TIMERANGE = 12  # hours
NEXT_ED_VISIT_TIMERANGE = 3  # days

# Define Data

In [5]:
# thin dicts to help organize input data
def get_df_dict(name: str, path: Path, lazy: bool = True) -> dict:
    """Get a dictionary of DataFrames from CSV files in a given path.
    DataFrames are LazyFrames if lazy is True, otherwise they are materialized in memory.
    """
    def read_csv_safe(f: Path, lazy: bool):
        try: 
            return pl.scan_csv(f) if lazy else pl.read_csv(f)
        except: 
            return pl.scan_csv(f, infer_schema=False) if lazy else pl.read_csv(f, infer_schema=False)
    return {
        f.name.split('.')[0]: read_csv_safe(f, lazy) 
        for f in [*path.glob("**/*.csv"), *path.glob("**/*.csv.gz")]
    }

core = get_df_dict("core", path / "core", lazy=True)
ed = get_df_dict("ed", path / "ed", lazy=True)
hosp = get_df_dict("hosp", path / "hosp", lazy=True)
icu = get_df_dict("icu", path / "icu", lazy=True)

# Basic Transformations

Data type conformations, select statements, etc.

## Prep `ed` Tables

In [6]:
ed['triage'] = (
    ed['triage']

    # alias vital signs columns in `triage`    
    .rename({
        v: 'triage_' + v for v in 
        ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity']
    })
    .with_columns(
        triage_pain = pl.col('triage_pain').map_elements(cd.convert_str_to_float, return_dtype=pl.Float64)
))

ed['vitalsign'] = (
    ed['vitalsign']

    # alias vital signs columns in `vitals`    
    .rename({
        v: 'ed_' + v for v in 
        ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'rhythm', 'pain']
    })
    .with_columns(
        ed_pain = pl.col('ed_pain').map_elements(cd.convert_str_to_float, return_dtype=pl.Float64)
))

ed['edstays'] = cd.conform_dates(ed['edstays'], ['intime', 'outtime'])
ed['vitalsign'] = cd.conform_dates(ed['vitalsign'], ['charttime'])


## Prep `core` Tables

In [7]:
# keep the same columns as the original `read_admissions_table`, renaming `race` -> `ethnicity` if needed
_adm = core['admissions']
if 'race' in _adm.collect_schema().names():
    _adm = _adm.rename({'race': 'ethnicity'})
_adm = _adm.select('subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'ethnicity', 'edregtime', 'edouttime', 'insurance')
core['admissions'] = cd.conform_dates(_adm, ['admittime', 'dischtime', 'deathtime'])

core['patients'] = cd.conform_dates(core['patients'], ['dod'])
core['patients'] = core['patients'].select(
    'subject_id', 'anchor_age', 'gender', 'anchor_year','dod'
)

## Prep `icu` Tables

In [8]:
icu['icustays'] = cd.conform_dates(icu['icustays'], ['intime', 'outtime'])

# Create Master Table

In [9]:
# join patient and admission data with ED stays
master_df = (
    ed['edstays'].join(core['patients'], on=['subject_id'], how='left')
    .join(core['admissions'], on=['subject_id', 'hadm_id'], how='left')
)

## Master Table Transformations

In [10]:
## Add Primitive Columns ##
master_df = master_df.with_columns(
    # add age
    in_year = pl.col('intime').dt.year(),
    age = pl.col('intime').dt.year() - pl.col('anchor_year') + pl.col('anchor_age'),

    # add inhospital mortality
    outcome_inhospital_mortality = (pl.col('dod').is_not_null() & (pl.col('dischtime') >= pl.col('dod'))).fill_null(False),

    # add ED length of stay
    ed_los = pl.col('outtime') - pl.col('intime'),
)

In [11]:
## Add Outcome ICU Transfer and Transfer Times ##
# earliest ICU `intime` per admission (from icustays, not the ED table)
_icu_first = icu['icustays'].group_by('subject_id', 'hadm_id').agg(intime_icu = pl.col('intime').min())
master_df = (
    master_df.join(_icu_first, on=['subject_id', 'hadm_id'], how='left')
    .with_columns(**{
        # duration of ICU transfer time        
        "time_to_icu_transfer": (pl.col('intime_icu') - pl.col('outtime')),

        # bool for ICU transfer within the specified time range
        f"outcome_icu_transfer_{ICU_TRANSFER_TIMERANGE}h": (
            (pl.col('intime_icu') - pl.col('outtime')) <= timedelta(hours=ICU_TRANSFER_TIMERANGE)
        ).fill_null(False),
    })
)

In [12]:
## Add Hospitalization and Critical Outcome Bools ##
master_df = master_df.with_columns(
    outcome_hospitalization = pl.col('hadm_id').is_not_null(),
    outcome_critical = pl.col('outcome_inhospital_mortality') | pl.col(f"outcome_icu_transfer_{ICU_TRANSFER_TIMERANGE}h")
)

In [13]:
## Null-Fill `ethnicity` Column ##

# fill with the subject's most common non-null value (ties -> earliest occurrence, as in the original),
# fall-back to 'UNKNOWN' if the subject has no non-null value
_subject_ethnicity = (
    master_df
    .filter(pl.col('ethnicity').is_not_null())
    .sort('subject_id', 'intime', 'stay_id')
    .with_row_index('_pos')
    .group_by('subject_id', 'ethnicity')
    .agg(_n=pl.len(), _first=pl.col('_pos').min())
    .sort('subject_id', '_n', '_first', descending=[False, True, False])
    .group_by('subject_id')
    .agg(_subject_ethnicity=pl.col('ethnicity').first())
)
master_df = (
    master_df
    .join(_subject_ethnicity, on='subject_id', how='left')
    .with_columns(
        ethnicity_was_null = pl.col('ethnicity').is_null(),  # flag if ethnicity was originally null
        ethnicity = pl.coalesce('ethnicity', '_subject_ethnicity', pl.lit('UNKNOWN')),
    )
    .drop('_subject_ethnicity')
)


## Master Table Transformations

### Health Utilization Computations

In [14]:
## Count Past ED Visits per Subject ##
# replicates `generate_past_ed_visits`

# count in [30, 90, 365] day look-back windows
# .rolling().over() is like a partitioned & sorted SQL window -- backwards looking by default
# period defines the range, closed specifies which endpoints are inclusive
master_df = master_df.with_columns(
    pl.len()
    .rolling(index_column='intime', period=f'{d}d', closed='left')  
    .over('subject_id', order_by='intime')
    .cast(pl.Int32)
    .alias(f'n_ed_{d}d')
    for d in [30, 90, 365]
)

In [15]:
## Get Time to Next ED Visit per Subject ##
# replicates `generate_future_ed_visits`

next_intime = pl.col('intime').shift(-1).over('subject_id', order_by=['intime', 'stay_id'])
gap = next_intime - pl.col('outtime')

master_df = master_df.with_columns(
    next_ed_visit_time = next_intime,
    next_ed_visit_time_diff = gap,
    **{f'outcome_ed_revisit_{NEXT_ED_VISIT_TIMERANGE}d': (gap < pl.duration(days=NEXT_ED_VISIT_TIMERANGE)).fill_null(False)},
)

In [16]:
## Count Past ICU Visits and Admissions per Subject ##
# replicates `generate_past_admissions` and `generate_past_icu_visits`

def add_prior_event_counts(master, events, event_time, days, out):
    """Count each master row's subject events with  intime - days <= event_time < intime."""
    stacked = pl.concat([
        master.select('subject_id', 'stay_id', t=pl.col('intime'), is_event=pl.lit(0, pl.Int32)),
        events.select('subject_id', stay_id=pl.lit(None, pl.Int64), t=pl.col(event_time),
                      is_event=pl.lit(1, pl.Int32)),
    ])
    counts = (
        stacked
        .with_columns(
            pl.col('is_event')
            .sum()
            .rolling(index_column='t', period=f'{days}d', closed='left')
            .over('subject_id', order_by='t')
            .alias(out)
        )
        .filter(pl.col('stay_id').is_not_null())          # keep only master rows
        .select('stay_id', pl.col(out).fill_null(0))
    )
    return master.join(counts, on='stay_id', how='left', maintain_order='left')


for D in (30, 90, 365):
    master_df = add_prior_event_counts(master_df, core['admissions'], 'admittime', D, f'n_hosp_{D}d')
    master_df = add_prior_event_counts(master_df, icu['icustays'], 'intime', D, f'n_icu_{D}d')

In [17]:
## Add Numeric Timedelta ##
HOUR, DAY = 3600, 86400

def duration_to(col, seconds_per_unit, decimals=2):
    return (pl.col(col).dt.total_seconds(fractional=True) / seconds_per_unit).round(decimals)

master_df = master_df.with_columns(
    ed_los_hours=duration_to('ed_los', HOUR),
    time_to_icu_transfer_hours=duration_to('time_to_icu_transfer', HOUR),
    next_ed_visit_time_diff_days=duration_to('next_ed_visit_time_diff', DAY),
)

### Triage Information

In [18]:
# merge triage_df onto master_df
master_df = master_df.join(ed['triage'], on=['subject_id', 'stay_id'], how='left')

# encode chief complaints
# simple one-hot binarizations from the chief complaints dict
complaint_dict = {
    "chiefcom_chest_pain" : "chest pain", 
    "chiefcom_abdominal_pain" : "abdominal pain|abd pain", 
    "chiefcom_headache" : "headache|lightheaded", 
    "chiefcom_shortness_of_breath" : "breath", 
    "chiefcom_back_pain" : "back pain", 
    "chiefcom_cough" : "cough", 
    "chiefcom_nausea_vomiting" : "nausea|vomit", 
    "chiefcom_fever_chills" : "fever|chill", 
    "chiefcom_syncope" :"syncope", 
    "chiefcom_dizziness" : "dizz"
}

master_df = master_df.with_columns(
    pl.col('chiefcomplaint')
        .str.contains(f'(?i){pattern}')  # (?i) makes the regex case-insensitive
        .fill_null(False)
        .alias(name)
    for name, pattern in complaint_dict.items()
)

### Comorbidities from Diagnosis ICD

In [19]:
# add comorbidity flags
master_df = mu.comorbidity(master_df, hosp['diagnoses_icd'], core['admissions'], days=365 * 5)

In [20]:
# ICD list datasets (for embedding models), written alongside the master dataset
OUTPUT_PATH = path / "outputs"
OUTPUT_PATH.mkdir(exist_ok=True)

for version in ('v9', 'v9_3digit', 'v10'):
    mu.extract_icd_list(ed['edstays'], hosp['diagnoses_icd'], core['admissions'], OUTPUT_PATH,
                        days=365 * 5, version=version)


ICD List written to:     C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_list_dataset_v9.parquet
Encoding map written to: C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_encode_map_v9.parquet
Number of unique ICD codes v9: 8664
ICD List written to:     C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_list_dataset_v9_3digit.parquet
Encoding map written to: C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_encode_map_v9_3digit.parquet
Number of unique ICD codes v9_3digit: 8609
ICD List written to:     C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_list_dataset_v10.parquet
Encoding map written to: C:\Users\benak\repos\mimic4ed-task\data\outputs\icd_encode_map_v10.parquet
Number of unique ICD codes v10: 14953


### ED Vital Signs

In [21]:
# last recorded (non-null) value per ED stay -- matches pandas `groupby().last()`
_vitals = ['ed_temperature', 'ed_heartrate', 'ed_resprate', 'ed_o2sat', 'ed_sbp', 'ed_dbp', 'ed_pain']

ed_vitals_last = (
    ed['vitalsign']
    .sort('charttime')
    .group_by('subject_id', 'stay_id')
    .agg(pl.col(v).drop_nulls().last().alias(f'{v}_last') for v in _vitals)
)
master_df = master_df.join(ed_vitals_last, on=['subject_id', 'stay_id'], how='left')


### Medication

In [ ]:
# distinct medications dispensed (`pyxis`) per ED stay; `gsn` falls back to `name` when missing
n_med = (
    ed['pyxis']
    .group_by('stay_id')
    .agg(n_med = pl.coalesce(pl.col('gsn').cast(pl.String), pl.col('name')).drop_nulls().n_unique())
)

# distinct reconciled medications (`medrecon`) per ED stay.
# NOTE: the original never applied its `gsn` <- `name` fallback here (the `fillna` result was discarded),
# so this counts distinct non-null `gsn` only, to match the original output.
n_medrecon = (
    ed['medrecon']
    .group_by('stay_id')
    .agg(n_medrecon = pl.col('gsn').drop_nulls().n_unique())
)

master_df = (
    master_df
    .join(n_med, on='stay_id', how='left')
    .join(n_medrecon, on='stay_id', how='left')
    .with_columns(pl.col('n_med', 'n_medrecon').fill_null(0))
)


# Output Master Dataset

In [23]:
master_df.collect().write_parquet(OUTPUT_PATH / "master_dataset.parquet")

<sys>:0: UserWarning: POLARS_MAX_BLOCKING_THREAD_COUNT reached (512), this may indicate a deadlock
